# Notebook consacré à l'analyse des variables nominales à choix unique




In [ ]:
import pandas as pd

import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket
import scipy.stats

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
list_affil = pd.read_csv("list_affiliation.csv", sep =",")
df0 = df0.merge(list_affil, on = ["q45_clé", "q44_ufr_labo"], how = "left")

In [ ]:
list_nominal_simple = [x for x in df_col.label.loc[(df_col.type.isin(["simple_nominal", "booléen","ordinal"]))]]


In [ ]:
def grouped_question(data, column, index = "q45_clé"):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_tmp = data.copy()
    gb_data = df_tmp.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data

In [ ]:
df0

In [ ]:
df_col["question_family"] = df_col.label.apply(lambda row : row.split("_")[0])

dict_question = dict(zip(df_col.label.loc[df_col.label.isin(list_nominal_simple)], df_col.question_family.loc[df_col.label.isin(list_nominal_simple)]))


# Types de données

In [ ]:
grouped_question(df0, column=col)

In [ ]:
%%capture cap
for n, family in enumerate(df_col.question_family.unique()):
    list_quest = [x for x in dict_question if dict_question[x]==family]
    if len(list_quest) > 0:
        question = df_col.question.loc[df_col.question_family == family].iloc[0]
        print("==================\n", question)
        for m, col in enumerate(list_quest):
            print(df_col.name.loc[df_col.label==col].iloc[0])
            gb_data = grouped_question(df0, column=col, index = "q45_clé")
            print(gb_data)
            print("---------------------\n")
            
        


In [ ]:
with open("tableau_frequence_question_simple.txt", 'w') as file_in:
    file_in.write(cap.stdout)

In [ ]:
def tab_croise(data, x, y, regroup_y = True, khideux = False) :
    """
    x = variable en ligne
    y = variable en colonne
    """
    if regroup_y == True:

        data.loc[data[y].str.lower().str.contains("oui"), f"{y}_rec"] = "Oui"
        data.loc[data[y].str.lower().str.contains("non"), f"{y}_rec"] = "Non"

        cross_tab = pd.crosstab(data[x], data[f"{y}_rec"], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[f"{y}_rec"], margins = False)
    else:
        cross_tab = pd.crosstab(data[x], data[y], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[f"{y}_rec"], margins = False)

    if khideux == True:
        print(scipy.stats.chi2_contingency(cross_tab0))
        st_chi2, st_p, st_dof, st_exp = scipy.stats.chi2_contingency(cross_tab0)
        chi2 = pd.DataFrame(data={"stats":["chi2","df","p-value"], "values":[st_chi2, st_dof, st_p]})
        cross_tab = pd.concat([cross_tab, chi2])
    else:
        pass

    return cross_tab.fillna("").reset_index()

In [ ]:
tab_croise(df0, x= "q25_statut_rec", y="q2_hal_depot", regroup_y = True, khideux = True)

In [ ]:
(15*107)/119

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
list_family = [x for x in dict_question.values()]
list_family_dedup = []
for x in list_family:
    if list_family_dedup.count(x) == 0:
        list_family_dedup.append(x)
    else:
        pass


In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(len(list_family), figsize=(10, 30))

# Plot the total crashes
sns.set_color_codes("pastel")

for n, family in enumerate(df_col.question_family.unique()):
    list_quest = [x for x in dict_question if dict_question[x]==family]
    if len(list_quest) > 0:
        for m, col in enumerate(list_quest):
            #print(col)
            gb_data = grouped_question(df0, column=col, index = "q45_clé")
        
            sns.barplot(x="freq", hue=col, data=gb_data, ax=ax[n])

       

In [ ]:
sns.barplot(x="nb", y=col, data=gb_data,
                label="effectif", color="r", ax=ax[n])